In [ ]:
import copy
import cv2
import glob
import math
import matplotlib.pyplot as plt
from matplotlib import colors
from mpl_toolkits.axes_grid1.anchored_artists import AnchoredSizeBar
import numpy as np
import os
import pandas as pd
import seaborn as sns
from astropy.io import fits
import torch
from torch.utils.data import Dataset
from torchmetrics import Dice
from torchmetrics import Dice, JaccardIndex
import segmentation_models_pytorch as smp

In [ ]:
hyperparameters = {
    'notebook_name': 'Evaluation_NISP_Y_1000imgs_v13_log_no_aug.ipynb',
    'experiment_name': 'Evaluation_NISP_Y_1000imgs_v13_log_no_aug_pad',
    'best_trained_model_name': 'NISP_Y_1000imgs_v13_log_no_aug_pad',
    'device': torch.device("cuda:0" if torch.cuda.is_available() else "cpu"),
    'galaxies_path': 'galaxies_test_NISP_Y/galaxy_and_stream_convolved*.fits',
    'masks_path': '../masks_train_NISP_Y/mask_',
    'original_image_width': 200,
    'original_image_height': 200,
    'target_image_width': 224,
    'target_image_height': 224
}

In [ ]:
def pad_to_size(image, target_h, target_w, pad_value=0):
    """Apply padding to reach target size without distortion"""
    h, w = image.shape[:2]
    pad_h = target_h - h
    pad_w = target_w - w
    
    # Symmetric padding
    pad_top = pad_h // 2
    pad_bottom = pad_h - pad_top
    pad_left = pad_w // 2
    pad_right = pad_w - pad_left
    
    if len(image.shape) == 2:  # For masks
        return np.pad(image, ((pad_top, pad_bottom), (pad_left, pad_right)), 
                     mode='constant', constant_values=pad_value)
    else:  # For RGB images
        return np.pad(image, ((pad_top, pad_bottom), (pad_left, pad_right), (0, 0)), 
                     mode='constant', constant_values=pad_value)

In [ ]:
def get_galaxy_number(galaxy_name: str):
    return os.path.basename(galaxy_name).split('_')[4]

def get_galaxy_magnitude(galaxy_name: str):
    return os.path.basename(galaxy_name).split('_')[5]

def normalize_01(inp: np.ndarray):
    """Squash image input to the value range [0, 1] (no clipping)"""
    inp_out = (inp - np.min(inp)) / np.ptp(inp)
    return inp_out

def logarithmic_image(img: np.ndarray) -> np.ndarray:
    """
    Apply logarithmic scaling to the image to increase contrast
    """
    img_log = np.log(img, where=(img > 0))
    min_value = np.min(img_log)
    np.putmask(img_log, img != 0, img_log + abs(min_value))
    return img_log

In [ ]:
def get_mag_stream_percentage(galaxy_number):
    
    catalogs_folder_path = "../../dataset_generation/catalogs_for_checking" 
    catalog_path = os.path.join(catalogs_folder_path, "stream_characteristics_" + str(galaxy_number) + ".txt")
    df = pd.read_csv(catalog_path, delimiter=r"\s+", engine="python")
    mag_bulge = df.iloc[1, 3]
    mag_disk = df.iloc[1, 8]
    mag_stream = df.iloc[1, 13]
    zeropoint = 30.132  # For VIS, (For NISP is 30)
    flux_bulge = 10 ** ((mag_bulge - zeropoint) / (-2.5)) 
    flux_disk = 10 ** ((mag_disk - zeropoint) / (-2.5))
    flux_galaxy = flux_bulge + flux_disk
    flux_stream = 10 ** ((mag_stream - zeropoint) / (-2.5)) 
    mag_stream_percentage = flux_stream / flux_galaxy
    mag_stream_percentage_rounded_up = math.ceil(mag_stream_percentage * 100)
    if mag_stream_percentage_rounded_up == 6:
        mag_stream_percentage_rounded_up = 5
    # print("MAG_STREAM_PERCENTAGE: " + str(mag_stream_percentage) + "->" + str(mag_stream_percentage_rounded_up) + "%")
    return mag_stream_percentage_rounded_up

In [ ]:
def get_distance_to_galaxy_center(galaxy_number):
    catalogs_folder_path = "../../dataset_generation/catalogs_for_checking" 
    catalog_path = os.path.join(catalogs_folder_path, "stream_characteristics_" + str(galaxy_number) + ".txt")
    df = pd.read_csv(catalog_path, delimiter=r"\s+", engine="python")
    distance_to_galaxy_center = df.iloc[1, 14]
    # print("RR_STREAM: " + str(distance_to_galaxy_center))
    return distance_to_galaxy_center

In [ ]:
class MyDataset(Dataset):
    """
    Basic dataset without any augmentation
    """
    def __init__(self, galaxies_with_tidal_streams, transform=None):
        """
        Dataset constructor
        @param galaxies_with_tidal_streams: List of paths to files containing galaxy data
        """
        self.galaxies_with_tidal_streams = galaxies_with_tidal_streams
        self.transform = transform

    def __getitem__(self, index):
        # Open the galaxy image with tidal stream
        path = self.galaxies_with_tidal_streams[index]
        actual_magnitude = get_galaxy_magnitude(path)
        
        galaxy_fits = fits.open(self.galaxies_with_tidal_streams[index])
        x = galaxy_fits[0].data.astype(np.float32)
        # Open the corresponding mask for the image
        galaxy_number = get_galaxy_number(self.galaxies_with_tidal_streams[index])
        galaxy_magnitude = get_galaxy_magnitude(self.galaxies_with_tidal_streams[index])
        mag_stream_percentage = get_mag_stream_percentage(galaxy_number)
        distance_to_galaxy_center = get_distance_to_galaxy_center(galaxy_number)
        y = np.array(cv2.imread(hyperparameters['masks_path'] + str(galaxy_number) + "_" + str(galaxy_magnitude) + ".png", 0)).astype(np.float32)
        # x = normalize_01(x)
        x_tensor_for_visualization = copy.deepcopy(x)
        x = logarithmic_image(x)

        x = normalize_01(x)

        x = pad_to_size(x, hyperparameters['target_image_height'], hyperparameters['target_image_width'])
        y = pad_to_size(y, hyperparameters['target_image_height'], hyperparameters['target_image_width'])
        if self.transform is not None:
            augmented = self.transform(image=x, mask=y)
            x_tensor = augmented["image"]
            y_tensor = augmented["mask"].long()
        else:
            x_tensor = torch.from_numpy(x).float()
            x_tensor = torch.unsqueeze(x_tensor, dim=0)
            y_tensor = torch.from_numpy(y).long()
        
        # Reshape the tensors
        y_tensor = torch.unsqueeze(y_tensor, dim=0)
        return x_tensor, y_tensor, path, actual_magnitude, x_tensor_for_visualization, mag_stream_percentage, distance_to_galaxy_center
    
    def __len__(self):
        """
        Returns the length of the dataset
        """
        return len(self.galaxies_with_tidal_streams)

In [ ]:
dice = Dice(num_classes=2, average='macro', ignore_index=0)
iou = JaccardIndex(task='multiclass', num_classes=2, average='none', ignore_index=0)
test_images = glob.glob(hyperparameters['galaxies_path'])
test_images.sort()
# Load the test dataset
test_dataset = MyDataset(test_images)
print("Number of valid images: " + str(len(test_images)))
device = hyperparameters['device']
# dice.to(device)

In [ ]:
unet = smp.Unet(
    encoder_name="resnet18",        # choose encoder, e.g. mobilenet_v2 or efficientnet-b7
    encoder_weights="imagenet",     # use `imagenet` pre-trained weights for encoder initialization
    in_channels=1,                  # model input channels (1 for gray-scale images, 3 for RGB, etc.)
    classes=2,                      # model output channels (number of classes in your dataset)
)

In [ ]:
# Load the previously trained model
model_path = hyperparameters['best_trained_model_name']
best_model = unet
best_model.load_state_dict(torch.load(model_path))
best_model.to(device)

In [ ]:
# ==========================================
# 1. DATA STRUCTURES DEFINITION
# ==========================================
dice_results = {
    '0.05': [], '0.1': [], '0.15': [], '0.2': [], '0.25': [],
    '0.4': [], '0.6': [], '0.8': [], '1': [],
}
iou_results = {
    '0.05': [], '0.1': [], '0.15': [], '0.2': [], '0.25': [],
    '0.4': [], '0.6': [], '0.8': [], '1': [],
}

dice_results_redshift_with_mag_stream_percentage = {
    '0.05': {1: [], 2: [], 3: [], 4: [], 5: []},
    '0.1': {1: [], 2: [], 3: [], 4: [], 5: []},
    '0.15': {1: [], 2: [], 3: [], 4: [], 5: []},
    '0.2': {1: [], 2: [], 3: [], 4: [], 5: []},
    '0.25': {1: [], 2: [], 3: [], 4: [], 5: []},
    '0.4': {1: [], 2: [], 3: [], 4: [], 5: []},
    '0.6': {1: [], 2: [], 3: [], 4: [], 5: []},
    '0.8': {1: [], 2: [], 3: [], 4: [], 5: []},
    '1': {1: [], 2: [], 3: [], 4: [], 5: []},
}

iou_results_redshift_with_mag_stream_percentage = {
    '0.05': {1: [], 2: [], 3: [], 4: [], 5: []},
    '0.1': {1: [], 2: [], 3: [], 4: [], 5: []},
    '0.15': {1: [], 2: [], 3: [], 4: [], 5: []},
    '0.2': {1: [], 2: [], 3: [], 4: [], 5: []},
    '0.25': {1: [], 2: [], 3: [], 4: [], 5: []},
    '0.4': {1: [], 2: [], 3: [], 4: [], 5: []},
    '0.6': {1: [], 2: [], 3: [], 4: [], 5: []},
    '0.8': {1: [], 2: [], 3: [], 4: [], 5: []},
    '1': {1: [], 2: [], 3: [], 4: [], 5: []},
}

# NEW STRUCTURES FOR DISTANCE
dice_results_redshift_with_distance = {
    '0.05': {}, '0.1': {}, '0.15': {}, '0.2': {}, '0.25': {},
    '0.4': {}, '0.6': {}, '0.8': {}, '1': {},
}

iou_results_redshift_with_distance = {
    '0.05': {}, '0.1': {}, '0.15': {}, '0.2': {}, '0.25': {},
    '0.4': {}, '0.6': {}, '0.8': {}, '1': {},
}

# ==========================================
# OUTPUT FOLDER FOR FITS FILES
# ==========================================
filter_name = "NISP_Y"
output_fits_folder = f"predictions_{filter_name}"
os.makedirs(output_fits_folder, exist_ok=True)
print(f"Output FITS folder created: {output_fits_folder}")

# ==========================================
# CALCULATE BINS AUTOMATICALLY
# ==========================================
print("Calculating distance range from dataset...")

# Collect all distances from the dataset
all_distances = []
for i in range(len(test_dataset)):
    item = test_dataset.__getitem__(i)
    all_distances.append(item[6])

# Calculate minimum and maximum
min_distance = min(all_distances)
max_distance = max(all_distances)

print(f"Minimum distance: {min_distance:.2f}")
print(f"Maximum distance: {max_distance:.2f}")

# Round to have integer limits
import math
min_distance_rounded = math.floor(min_distance)  # Round down
max_distance_rounded = math.ceil(max_distance)   # Round up

# Create 5 equal intervals
num_bins = 5
bin_width = (max_distance_rounded - min_distance_rounded) / num_bins

# Generate the bins
distance_bins = [min_distance_rounded + i * bin_width for i in range(num_bins + 1)]
# Round each bin to integer
distance_bins = [round(b) for b in distance_bins]

# Create the labels
distance_labels = [f"{distance_bins[i]}-{distance_bins[i+1]}" for i in range(num_bins)]

print(f"\nDistance bins created: {distance_bins}")
print(f"Labels: {distance_labels}")

# ==========================================
# 2. EVALUATION ON TEST DATASET (with visualization)
# ==========================================
width = hyperparameters['target_image_width']
height = hyperparameters['target_image_height']
best_model.eval()
test_steps = 0
mean_dice_test = 0
mean_iou_test = 0
zp = 29.9
pix_scale = 0.3

for i in range(int(len(test_dataset) * 0.01)):
    dataset_item = test_dataset.__getitem__(i)
    item_x = dataset_item[0].reshape((1, 1, width, height)).cpu().detach().squeeze().numpy()
    item_label = dataset_item[1].reshape((1, 1, width, height)).cpu().detach().squeeze().numpy()
    x_tensor_for_visualization = dataset_item[0].reshape((1, 1, width, height)).cpu().detach().squeeze().numpy()
    
    # Get the original FITS file path
    original_fits_path = dataset_item[2]
    
    # If the mask is empty, we don't try to segment it
    if np.count_nonzero(item_label) != 0:
        # Get network prediction
        prediction_tensor = best_model(dataset_item[0].reshape((1, 1, width, height)).to(device)).cpu().detach().squeeze()
        predicted_mask = torch.argmax(prediction_tensor, dim=0).numpy()
        dice_value = dice(torch.from_numpy(item_label), torch.from_numpy(predicted_mask))
        iou_value = iou(torch.from_numpy(item_label), torch.from_numpy(predicted_mask))[1]
        
        magnitude = dataset_item[3]
        mag_stream_percentage = dataset_item[5]
        distance_to_galaxy_center = dataset_item[6]
        
        print("Path: " + dataset_item[2] + "\nMagnitude: " + magnitude + 
              ", Dice: " + str(round(dice_value.item(), 4)) + " IoU: " + str(round(iou_value.item(), 4)) +
              ", Distance: " + str(round(distance_to_galaxy_center, 2)))
        
        # ==========================================
        # SAVE FITS FILE WITH 4 CHANNELS
        # ==========================================
        with fits.open(original_fits_path) as original_hdu:
            original_header = original_hdu[0].header.copy()
        
        channel_1 = prediction_tensor[0].numpy()
        channel_2 = prediction_tensor[1].numpy()
        binary_mask = predicted_mask.astype(np.float32)
        channel_diff = channel_1 - channel_2
        
        multi_channel_data = np.stack([channel_1, channel_2, binary_mask, channel_diff], axis=0)
        
        original_header['NAXIS'] = 3
        original_header['NAXIS3'] = 4
        original_header['CHAN1'] = ('BKG_LOGIT', 'Background class logits')
        original_header['CHAN2'] = ('STR_LOGIT', 'Stream class logits')
        original_header['CHAN3'] = ('BIN_MASK', 'Binary predicted mask')
        original_header['CHAN4'] = ('LOGIT_DIF', 'Difference channel1 - channel2')
        original_header['FILTER'] = (filter_name, 'Euclid filter used')
        original_header['DICE'] = (round(dice_value.item(), 4), 'Dice score')
        original_header['IOU'] = (round(iou_value.item(), 4), 'IoU score')
        
        original_basename = os.path.basename(original_fits_path)
        output_filename = original_basename.replace('.fits', f'_prediction.fits')
        output_path = os.path.join(output_fits_folder, output_filename)
        
        hdu = fits.PrimaryHDU(data=multi_channel_data, header=original_header)
        hdu.writeto(output_path, overwrite=True)
        
        print(f"  -> Saved prediction FITS: {output_path}")
        
        # Store in existing structures
        dice_results[magnitude].append(dice_value.item())
        iou_results[magnitude].append(iou_value.item())
        dice_results_redshift_with_mag_stream_percentage[magnitude][mag_stream_percentage].append(dice_value.item())
        iou_results_redshift_with_mag_stream_percentage[magnitude][mag_stream_percentage].append(iou_value.item())
        
        # NEW: Determine distance bin and store
        distance_bin_idx = None
        for idx in range(len(distance_bins) - 1):
            if distance_bins[idx] <= distance_to_galaxy_center < distance_bins[idx + 1]:
                distance_bin_idx = idx
                break
        
        # Special case: if it's exactly the maximum value
        if distance_to_galaxy_center == distance_bins[-1]:
            distance_bin_idx = len(distance_bins) - 2
        
        if distance_bin_idx is not None:
            distance_label = distance_labels[distance_bin_idx]
            
            # Initialize if it doesn't exist
            if distance_label not in dice_results_redshift_with_distance[magnitude]:
                dice_results_redshift_with_distance[magnitude][distance_label] = []
                iou_results_redshift_with_distance[magnitude][distance_label] = []
            
            # Add results
            dice_results_redshift_with_distance[magnitude][distance_label].append(dice_value.item())
            iou_results_redshift_with_distance[magnitude][distance_label].append(iou_value.item())
        
        mean_dice_test += dice_value.item()
        mean_iou_test += iou_value.item()
        test_steps += 1
        
        # Display image, mask, and predicted mask
        item_x = item_x.astype(np.float32)   
        item_label = item_label.astype(np.uint8)
        predicted_mask = predicted_mask.astype(np.float32)  

        # Remove padding when displaying
        crop = 12
        item_x = item_x[crop:-crop, crop:-crop]
        item_label = item_label[crop:-crop, crop:-crop]
        predicted_mask = predicted_mask[crop:-crop, crop:-crop]

        x_tensor_for_visualization = x_tensor_for_visualization[crop:-crop, crop:-crop]
        zeros = np.zeros((hyperparameters['original_image_width'], hyperparameters['original_image_width']))
        ones = np.ones((hyperparameters['original_image_width'], hyperparameters['original_image_width']))

        cmap = colors.ListedColormap(['rebeccapurple', 'yellow', 'crimson', 'lime'])
        bounds = [0, 1, 2, 3, 4]
        norm = colors.BoundaryNorm(bounds, cmap.N)
        diff = np.zeros((hyperparameters['original_image_width'], hyperparameters['original_image_width']))
        diff[(item_label == zeros) & (predicted_mask == zeros)] = 0
        diff[(item_label == ones) & (predicted_mask == ones)] = 1
        diff[(item_label == zeros) & (predicted_mask == ones)] = 2
        diff[(item_label == ones) & (predicted_mask == zeros)] = 3

        fig, (axs0, axs1, axs2, axs3) = plt.subplots(1, 4, figsize=(15, 15))

        sb_array = -2.5 * np.log10(x_tensor_for_visualization) + zp + (5 * np.log10(pix_scale))

        # Filter inf and nan to calculate limits
        sb_valid = sb_array[np.isfinite(sb_array)]
        min_sb = np.min(sb_valid)
        max_sb = np.max(sb_valid)

        print("min_sb: " + str(min_sb))
        print("max_sb: " + str(max_sb))
        imshow_img = axs0.imshow(sb_array, origin='lower', interpolation="none", cmap="viridis_r", vmin=max_sb, vmax=min_sb)
        cb = fig.colorbar(imshow_img, ax=axs0, fraction=0.046, pad=0.04)
        cb.set_label(r'Surface brightness [mag/arcsec$^{2}$]')

        scalebar = AnchoredSizeBar(axs0.transData, 10. / pix_scale, "10 arcsec", borderpad=1.0, sep=5, loc="lower right", pad=0.5, color="blue", frameon=True, size_vertical=3)
        axs0.add_artist(scalebar)

        axs1.imshow(item_label, interpolation='none', origin="lower")
        axs2.imshow(predicted_mask, interpolation='none', origin="lower", cmap="plasma")
        axs2.text(0.05, 0.95, "Dice: " + str(round(dice_value.item(), 4)), color='white', fontsize=20, ha='left', va='top', transform=axs2.transAxes)
        axs3.imshow(diff, interpolation='none', origin="lower", cmap=cmap, norm=norm)
        axs3.text(0.05, 0.95, "IoU: " + str(round(iou_value.item(), 4)), color='white', fontsize=20, ha='left', va='top', transform=axs3.transAxes)
        plt.subplots_adjust(wspace=0.7)
        plt.show()

In [ ]:
# ==========================================
# 1. DATA STRUCTURES DEFINITION
# ==========================================
dice_results = {
    '0.05': [], '0.1': [], '0.15': [], '0.2': [], '0.25': [],
    '0.4': [], '0.6': [], '0.8': [], '1': [],
}
iou_results = {
    '0.05': [], '0.1': [], '0.15': [], '0.2': [], '0.25': [],
    '0.4': [], '0.6': [], '0.8': [], '1': [],
}

dice_results_redshift_with_mag_stream_percentage = {
    '0.05': {1: [], 2: [], 3: [], 4: [], 5: []},
    '0.1': {1: [], 2: [], 3: [], 4: [], 5: []},
    '0.15': {1: [], 2: [], 3: [], 4: [], 5: []},
    '0.2': {1: [], 2: [], 3: [], 4: [], 5: []},
    '0.25': {1: [], 2: [], 3: [], 4: [], 5: []},
    '0.4': {1: [], 2: [], 3: [], 4: [], 5: []},
    '0.6': {1: [], 2: [], 3: [], 4: [], 5: []},
    '0.8': {1: [], 2: [], 3: [], 4: [], 5: []},
    '1': {1: [], 2: [], 3: [], 4: [], 5: []},
}

iou_results_redshift_with_mag_stream_percentage = {
    '0.05': {1: [], 2: [], 3: [], 4: [], 5: []},
    '0.1': {1: [], 2: [], 3: [], 4: [], 5: []},
    '0.15': {1: [], 2: [], 3: [], 4: [], 5: []},
    '0.2': {1: [], 2: [], 3: [], 4: [], 5: []},
    '0.25': {1: [], 2: [], 3: [], 4: [], 5: []},
    '0.4': {1: [], 2: [], 3: [], 4: [], 5: []},
    '0.6': {1: [], 2: [], 3: [], 4: [], 5: []},
    '0.8': {1: [], 2: [], 3: [], 4: [], 5: []},
    '1': {1: [], 2: [], 3: [], 4: [], 5: []},
}

# NEW STRUCTURES FOR DISTANCE
dice_results_redshift_with_distance = {
    '0.05': {}, '0.1': {}, '0.15': {}, '0.2': {}, '0.25': {},
    '0.4': {}, '0.6': {}, '0.8': {}, '1': {},
}

iou_results_redshift_with_distance = {
    '0.05': {}, '0.1': {}, '0.15': {}, '0.2': {}, '0.25': {},
    '0.4': {}, '0.6': {}, '0.8': {}, '1': {},
}

# ==========================================
# OUTPUT FOLDER FOR FITS FILES
# ==========================================
filter_name = "NISP_Y"
output_fits_folder = f"predictions_{filter_name}"
os.makedirs(output_fits_folder, exist_ok=True)
print(f"Output FITS folder created: {output_fits_folder}")

# ==========================================
# CALCULATE BINS AUTOMATICALLY
# ==========================================
print("Calculating distance range from dataset...")

# Collect all distances from the dataset
all_distances = []
for i in range(len(test_dataset)):
    item = test_dataset.__getitem__(i)
    all_distances.append(item[6])

# Calculate minimum and maximum
min_distance = min(all_distances)
max_distance = max(all_distances)

print(f"Minimum distance: {min_distance:.2f}")
print(f"Maximum distance: {max_distance:.2f}")

# Round to have integer limits
import math
min_distance_rounded = math.floor(min_distance)  # Round down
max_distance_rounded = math.ceil(max_distance)   # Round up

# Create 5 equal intervals
num_bins = 5
bin_width = (max_distance_rounded - min_distance_rounded) / num_bins

# Generate the bins
distance_bins = [min_distance_rounded + i * bin_width for i in range(num_bins + 1)]
# Round each bin to integer
distance_bins = [round(b) for b in distance_bins]

# Create the labels
distance_labels = [f"{distance_bins[i]}-{distance_bins[i+1]}" for i in range(num_bins)]

print(f"\nDistance bins created: {distance_bins}")
print(f"Labels: {distance_labels}")

# ==========================================
# 2. EVALUATION ON TEST DATASET
# ==========================================
width = hyperparameters['target_image_width']
height = hyperparameters['target_image_height']
best_model.eval()
test_steps = 0
mean_dice_test = 0
mean_iou_test = 0
zp = 29.9
pix_scale = 0.3

for i in range(int(len(test_dataset))):
    dataset_item = test_dataset.__getitem__(i)
    item_x = dataset_item[0].reshape((1, 1, width, height)).cpu().detach().squeeze().numpy()
    item_label = dataset_item[1].reshape((1, 1, width, height)).cpu().detach().squeeze().numpy()
    x_tensor_for_visualization = dataset_item[0].reshape((1, 1, width, height)).cpu().detach().squeeze().numpy()
    
    # Get the original FITS file path
    original_fits_path = dataset_item[2]
    
    # If the mask is empty, we don't try to segment it
    if np.count_nonzero(item_label) != 0:
        # Get network prediction
        prediction_tensor = best_model(dataset_item[0].reshape((1, 1, width, height)).to(device)).cpu().detach().squeeze()   
        predicted_mask = torch.argmax(prediction_tensor, dim=0).numpy()
        dice_value = dice(torch.from_numpy(item_label), torch.from_numpy(predicted_mask))
        iou_value = iou(torch.from_numpy(item_label), torch.from_numpy(predicted_mask))[1]
        
        magnitude = dataset_item[3]
        mag_stream_percentage = dataset_item[5]
        distance_to_galaxy_center = dataset_item[6]
        
        print("Path: " + dataset_item[2] + "\nMagnitude: " + magnitude + 
              ", Dice: " + str(round(dice_value.item(), 4)) + " IoU: " + str(round(iou_value.item(), 4)) +
              ", Distance: " + str(round(distance_to_galaxy_center, 2)))
        
        # ==========================================
        # SAVE FITS FILE WITH 4 CHANNELS
        # ==========================================
        with fits.open(original_fits_path) as original_hdu:
            original_header = original_hdu[0].header.copy()
        
        channel_1 = prediction_tensor[0].numpy()
        channel_2 = prediction_tensor[1].numpy()
        binary_mask = predicted_mask.astype(np.float32)
        channel_diff = channel_1 - channel_2
        
        multi_channel_data = np.stack([channel_1, channel_2, binary_mask, channel_diff], axis=0)
        
        original_header['NAXIS'] = 3
        original_header['NAXIS3'] = 4
        original_header['CHAN1'] = ('BKG_LOGIT', 'Background class logits')
        original_header['CHAN2'] = ('STR_LOGIT', 'Stream class logits')
        original_header['CHAN3'] = ('BIN_MASK', 'Binary predicted mask')
        original_header['CHAN4'] = ('LOGIT_DIF', 'Difference channel1 - channel2')
        original_header['FILTER'] = (filter_name, 'Euclid filter used')
        original_header['DICE'] = (round(dice_value.item(), 4), 'Dice score')
        original_header['IOU'] = (round(iou_value.item(), 4), 'IoU score')
        
        original_basename = os.path.basename(original_fits_path)
        output_filename = original_basename.replace('.fits', f'_prediction.fits')
        output_path = os.path.join(output_fits_folder, output_filename)
        
        hdu = fits.PrimaryHDU(data=multi_channel_data, header=original_header)
        hdu.writeto(output_path, overwrite=True)
        
        print(f"  -> Saved prediction FITS: {output_path}")
        
        # Store in existing structures
        dice_results[magnitude].append(dice_value.item())
        iou_results[magnitude].append(iou_value.item())
        dice_results_redshift_with_mag_stream_percentage[magnitude][mag_stream_percentage].append(dice_value.item())
        iou_results_redshift_with_mag_stream_percentage[magnitude][mag_stream_percentage].append(iou_value.item())
        
        # NEW: Determine distance bin and store
        distance_bin_idx = None
        for idx in range(len(distance_bins) - 1):
            if distance_bins[idx] <= distance_to_galaxy_center < distance_bins[idx + 1]:
                distance_bin_idx = idx
                break
        
        # Special case: if it's exactly the maximum value
        if distance_to_galaxy_center == distance_bins[-1]:
            distance_bin_idx = len(distance_bins) - 2
        
        if distance_bin_idx is not None:
            distance_label = distance_labels[distance_bin_idx]
            
            # Initialize if it doesn't exist
            if distance_label not in dice_results_redshift_with_distance[magnitude]:
                dice_results_redshift_with_distance[magnitude][distance_label] = []
                iou_results_redshift_with_distance[magnitude][distance_label] = []
            
            # Add results
            dice_results_redshift_with_distance[magnitude][distance_label].append(dice_value.item())
            iou_results_redshift_with_distance[magnitude][distance_label].append(iou_value.item())
        
        mean_dice_test += dice_value.item()
        mean_iou_test += iou_value.item()
        test_steps += 1

print(f"\n{'='*50}")
print(f"Evaluation complete. {test_steps} FITS files saved to '{output_fits_folder}/'")
print(f"{'='*50}")

In [ ]:
total_mean_dice_test = mean_dice_test / test_steps
print("Total_mean_Dice=" + str(total_mean_dice_test))

In [ ]:
total_mean_iou_test = mean_iou_test / test_steps
print("Total_mean_IoU=" + str(total_mean_iou_test))

In [ ]:
# Filter NaNs and calculate mean for dice_results
mean_dice_by_magnitude = [np.mean([x for x in lst if not np.isnan(x)]) for lst in dice_results.values()]
print(mean_dice_by_magnitude)
# Filter NaNs and calculate mean for iou_results
mean_iou_by_magnitude = [np.mean([x for x in lst if not np.isnan(x)]) for lst in iou_results.values()]
print(mean_iou_by_magnitude)

In [ ]:
# Create a color palette based on the achieved values, normalized between 0 and 1
norm = plt.Normalize(0, 1)
sm = plt.cm.ScalarMappable(cmap="magma", norm=norm)
sm.set_array([])

# Adjust the figure size
plt.figure(figsize=(12, 8))

# Create the bar chart
bar_colors = [sm.to_rgba(value) for value in mean_dice_by_magnitude]
ax = sns.barplot(x=list(dice_results.keys()), y=mean_dice_by_magnitude, palette=bar_colors)

# Adjust y-axis limits between 0 and 1
ax.set_ylim(0, 1)

# Add labels with values above the bars
for i, value in enumerate(mean_dice_by_magnitude):
    ax.text(i, value + 0.01, f'{value:.2f}', ha='center', va='bottom', fontsize=12, fontweight='bold', color='black')

# Add title and adjust labels
plt.title('NISP H Filter', fontsize=20, fontweight='bold')
plt.xlabel('Redshift', fontsize=16)
plt.ylabel('Dice', fontsize=16)

# Adjust axis label sizes
plt.xticks(fontsize=14)
plt.yticks(fontsize=14)

# Add grid lines
ax.grid(True, linestyle='--', linewidth=0.5)

# Add colorbar
cbar = plt.colorbar(sm, ax=ax)

# Improve overall design
sns.despine()
# plt.savefig('images_paper/nisp_h_filter_dice.png', dpi=300, bbox_inches='tight')
# Show the plot
plt.show()

In [ ]:
# Create a color palette based on the achieved values, normalized between 0 and 1
norm = plt.Normalize(0, 1)
sm = plt.cm.ScalarMappable(cmap="magma", norm=norm)
sm.set_array([])

# Adjust the figure size
plt.figure(figsize=(12, 8))

# Create the bar chart
bar_colors = [sm.to_rgba(value) for value in mean_iou_by_magnitude]
ax = sns.barplot(x=list(dice_results.keys()), y=mean_iou_by_magnitude, palette=bar_colors)

# Adjust y-axis limits between 0 and 1
ax.set_ylim(0, 1)

# Add labels with values above the bars
for i, value in enumerate(mean_iou_by_magnitude):
    ax.text(i, value + 0.01, f'{value:.2f}', ha='center', va='bottom', fontsize=12, fontweight='bold', color='black')

# Add title and adjust labels
plt.title('NISP H Filter', fontsize=20, fontweight='bold')
plt.xlabel('Redshift', fontsize=16)
plt.ylabel('IoU', fontsize=16)

# Adjust axis label sizes
plt.xticks(fontsize=14)
plt.yticks(fontsize=14)

# Add grid lines
ax.grid(True, linestyle='--', linewidth=0.5)

# Add colorbar
cbar = plt.colorbar(sm, ax=ax)

# Improve overall design
sns.despine()
# plt.savefig('images_paper/nisp_h_filter_iou.png', dpi=300, bbox_inches='tight')
# Show the plot
plt.show()

In [ ]:
dice_matrix_data = {
    redshift: {key: np.nanmean(values) for key, values in subdict.items()} 
    for redshift, subdict in dice_results_redshift_with_mag_stream_percentage.items()
}

redshift_labels = ['0.05', '0.1', '0.15', '0.2', '0.25', '0.4', '0.6', '0.8', '1']
mag_percentage_labels = ['1%', '2%', '3%', '4%', '5%']

# Convert data to a Pandas DataFrame
matrix_for_df = []
for redshift in redshift_labels:
    row = []
    for i, mag_label in enumerate(mag_percentage_labels, 1):  # 1, 2, 3, 4, 5
        if redshift in dice_matrix_data and i in dice_matrix_data[redshift]:
            row.append(dice_matrix_data[redshift][i])
        else:
            row.append(np.nan)
    matrix_for_df.append(row)

# Create DataFrame
df = pd.DataFrame(matrix_for_df, index=redshift_labels, columns=mag_percentage_labels)

print(df)
# Create heatmap with annotations
plt.figure(figsize=(8, 6))
sns.heatmap(df, annot=True, fmt=".3f", cmap="magma", linewidths=0.5,
           cbar_kws={'label': 'Dice'}, vmin=0, vmax=1.0)

plt.xlabel('Stream magnitude over galaxy magnitude percentage')
plt.ylabel('Redshift')
plt.title("NISP H Dice performance")
plt.tight_layout()
plt.show()

In [ ]:
iou_matrix_data = {
    redshift: {key: np.nanmean(values) for key, values in subdict.items()} 
    for redshift, subdict in iou_results_redshift_with_mag_stream_percentage.items()
}

redshift_labels = ['0.05', '0.1', '0.15', '0.2', '0.25', '0.4', '0.6', '0.8', '1']
mag_percentage_labels = ['1%', '2%', '3%', '4%', '5%']

# Convert data to a Pandas DataFrame
matrix_for_df = []
for redshift in redshift_labels:
    row = []
    for i, mag_label in enumerate(mag_percentage_labels, 1):  # 1, 2, 3, 4, 5
        if redshift in iou_matrix_data and i in iou_matrix_data[redshift]:
            row.append(iou_matrix_data[redshift][i])
        else:
            row.append(np.nan)
    matrix_for_df.append(row)

# Create DataFrame
df = pd.DataFrame(matrix_for_df, index=redshift_labels, columns=mag_percentage_labels)

print(df)
# Create heatmap with annotations
plt.figure(figsize=(8, 6))
sns.heatmap(df, annot=True, fmt=".3f", cmap="magma", linewidths=0.5,
           cbar_kws={'label': 'IoU'}, vmin=0, vmax=1.0)

plt.xlabel('Stream magnitude over galaxy magnitude percentage')
plt.ylabel('Redshift')
plt.title("NISP H Intersection over Union performance")
plt.tight_layout()
plt.show()

In [ ]:
dice_distance_matrix_data = {
    redshift: {key: np.nanmean(values) for key, values in subdict.items()} 
    for redshift, subdict in dice_results_redshift_with_distance.items()
}

distance_matrix_for_df = []
for redshift in redshift_labels:
    row = []
    for distance_label in distance_labels:
        if redshift in dice_distance_matrix_data and distance_label in dice_distance_matrix_data[redshift]:
            row.append(dice_distance_matrix_data[redshift][distance_label])
        else:
            row.append(np.nan)
    distance_matrix_for_df.append(row)

df_distance = pd.DataFrame(distance_matrix_for_df, 
                          index=redshift_labels, 
                          columns=distance_labels)

print("\n=== Dice vs Distance to Galaxy Center ===")
print(df_distance)

plt.figure(figsize=(10, 6))
sns.heatmap(df_distance, annot=True, fmt=".3f", cmap="magma", 
           linewidths=0.5, cbar_kws={'label': 'Dice'}, 
           vmin=0, vmax=1.0)
plt.xlabel('Distance to Galaxy Center (pixels)')
plt.ylabel('Redshift')
plt.title("NISP H Dice performance vs Distance to Galaxy Center")
plt.tight_layout()
plt.show()

In [ ]:
iou_distance_matrix_data = {
    redshift: {key: np.nanmean(values) for key, values in subdict.items()} 
    for redshift, subdict in iou_results_redshift_with_distance.items()
}

iou_distance_matrix_for_df = []
for redshift in redshift_labels:
    row = []
    for distance_label in distance_labels:
        if redshift in iou_distance_matrix_data and distance_label in iou_distance_matrix_data[redshift]:
            row.append(iou_distance_matrix_data[redshift][distance_label])
        else:
            row.append(np.nan)
    iou_distance_matrix_for_df.append(row)

df_iou_distance = pd.DataFrame(iou_distance_matrix_for_df, 
                              index=redshift_labels, 
                              columns=distance_labels)

print("\n=== IoU vs Distance to Galaxy Center ===")
print(df_iou_distance)

plt.figure(figsize=(10, 6))
sns.heatmap(df_iou_distance, annot=True, fmt=".3f", cmap="magma", 
           linewidths=0.5, cbar_kws={'label': 'IoU'}, 
           vmin=0, vmax=1.0)
plt.xlabel('Distance to Galaxy Center (pixels)')
plt.ylabel('Redshift')
plt.title("NISP H IoU performance vs Distance to Galaxy Center")
plt.tight_layout()
plt.show()